# Day 12 · Power — ใครถือครองฟ้ามืดของไทย

**#30DayMapChallenge 2026 · วันที่ 12 พ.ย. 2569** · โจทย์: *"Power as in nuclear power? Or physical
power or influence? Power grid on a map? Read this the way you want."*

อ่านทาง **"อำนาจ"** — ฟิลด์ `type` ของทะเบียน NARIT เป็นสภาพการถือครองที่ดินโดยตัวมันเอง
ผล: **43 จาก 83 จุด (51.8%) เป็นที่ดินส่วนบุคคล** ซึ่งให้บริการ **688 จาก 1,738 hex (39.6%)
= 197,595 ตร.กม. หรือ 38.6% ของประเทศ** และมี **11 จังหวัดที่พึ่งที่ดินเอกชน 100%**

> บน Colab: สคริปต์อ่าน geojson/CSV จาก GitHub ได้โดยแก้ตัวแปร `REPO`


### ตั้งค่า + helper (guards, สี, ชื่อประเภทแบบย่อ)

In [ ]:
# -*- coding: utf-8 -*-
"""
Day 12 · Power — "Power as in nuclear power? Or physical power or influence? Power grid on a map?"

ผมเลือกอ่านทาง "อำนาจ" และถามคำถามตรง ๆ: *ใครถือครองฟ้ามืดของไทย*
ทะเบียน NARIT จำแนกจุดด้วยฟิลด์ type ซึ่งโดยเนื้อแท้คือ "สภาพการถือครองที่ดิน"
(ส่วนบุคคล / อุทยานท้องฟ้ามืด / ชานเมือง / ชุมชน / อุทยานแห่งชาติ)

สองตัวเลขที่ต้องไม่พูดปนกัน:
  • ส่วนแบ่ง "จำนวนจุด"  → เอกชน 43 จาก 83 = 51.8%
  • ส่วนแบ่ง "พื้นที่ประเทศที่จุดนั้นให้บริการ" → เอกชน 688 จาก 1,738 hex = 39.6%
"""
import os, sys, json, time
import numpy as np, pandas as pd, geopandas as gpd
import shapely
from shapely.geometry import shape, Point, mapping
from shapely.ops import unary_union
from pyproj import CRS
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.collections import PolyCollection

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from guards import guard_bbox, guard_area, guard_variance, GuardError

T0 = time.time()
WORK = r"D:\GIS30Days"; REPO = os.path.join(WORK, "geo-notes")
OLD = os.path.join(WORK, "cache", "day06"); OUT = os.path.join(WORK, "out", "day12")
os.makedirs(OUT, exist_ok=True)
for f in sorted(os.listdir(OLD)):
    if f.endswith(".ttf"): font_manager.fontManager.addfont(os.path.join(OLD, f))
plt.rcParams["font.family"] = "Trirong"
log = lambda m: print(f"[{time.time()-T0:5.1f}s] {m}", flush=True)

EQ = CRS.from_proj4("+proj=eqdc +lat_0=14 +lon_0=101 +lat_1=8 +lat_2=20 +units=m +no_defs +ellps=WGS84")
BG, INK = "#0d1522", "#e8e3d3"
SHORT = {"เขตอนุรักษ์ท้องฟ้ามืดในพื้นที่ส่วนบุคคล": "ที่ดินส่วนบุคคล",
         "อุทยานท้องฟ้ามืด": "อุทยานท้องฟ้ามืด",
         "เขตอนุรักษ์ท้องฟ้ามืดในพื้นที่ชานเมือง": "ชานเมือง",
         "ชุมชนอนุรักษ์ท้องฟ้ามืด": "ชุมชน",
         "อุทยานแห่งชาติ": "อุทยานแห่งชาติ",
         "เขตอนุรักษ์ท้องฟ้ามืดในพื้นที่ส่วนบุคคลเขตชุมชน": "ส่วนบุคคล+ชุมชน"}
COL = {"ที่ดินส่วนบุคคล": "#f4566a", "อุทยานท้องฟ้ามืด": "#38bdf8", "ชานเมือง": "#a78bfa",
       "ชุมชน": "#2dd4bf", "อุทยานแห่งชาติ": "#f5b942", "ส่วนบุคคล+ชุมชน": "#e879f9"}

### 1. ข้อมูล + join

In [ ]:
df = pd.read_csv(os.path.join(REPO, "data/darksky_thailand.csv"))
df["name"] = df["name"].astype(str).str.strip()
lut_type = dict(zip(df["name"], df["type"].astype(str)))
lut_type = {k: SHORT.get(v, v) for k, v in lut_type.items()}
lut_region = dict(zip(df["name"], df["region_th"].astype(str)))
g3 = json.load(open(os.path.join(REPO, "posts/day03-polygons/day03_hex_polygons.geojson"), encoding="utf-8"))
hexes = [shape(f["geometry"]) for f in g3["features"]]
km_dark = np.array([f["properties"]["km_nearest"] for f in g3["features"]], float)
near = np.array([str(f["properties"]["nearest"]).strip() for f in g3["features"]])
missing = [n for n in set(near) if n not in lut_type]
assert not missing, f"ชื่อจุดใน hex ไม่พบใน CSV: {missing[:3]}"
hex_type = np.array([lut_type[n] for n in near])
hex_region = np.array([lut_region[n] for n in near])
cen = gpd.GeoSeries(hexes, crs="EPSG:4326").to_crs(EQ)
cent = np.array([[g.centroid.x, g.centroid.y] for g in cen])
cent_deg = np.array([[h.centroid.x, h.centroid.y] for h in hexes])
guard_bbox(cent_deg, 96.5, 107.6, 4.3, 21.7, "hex-centroid-degree")
log(f"hex {len(hexes)} · จุดที่ให้บริการจริง {len(set(near))} จาก {len(df)} จุด · join ครบ 100%")

### 2. ตรวจความสอดคล้องของฟิลด์ nearest

In [ ]:
def hav(a, b):
    la1, lo1 = np.radians(a[:, 1][:, None]), np.radians(a[:, 0][:, None])
    la2, lo2 = np.radians(b[:, 1][None, :]), np.radians(b[:, 0][None, :])
    d = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(d))
sites = np.column_stack([df.longitude.values, df.latitude.values])
names = df["name"].values
recheck = names[hav(cent_deg, sites).argmin(axis=1)]
agree = float((recheck == near).mean())
log(f"ตรวจซ้ำด้วยพิกัด: nearest ตรงกับที่ไฟล์บันทึก {agree*100:.2f}% "
    f"({int((recheck != near).sum())} เซลล์ต่าง — เหตุจาก polygon centroid ≠ H3 cell centre ของเซลล์ที่ถูก clip)")

### 3. ส่วนแบ่งสองแบบ (ห้ามพูดปนกัน)

In [ ]:
n_by_type = df["type"].astype(str).map(SHORT).value_counts()
hex_by_type = pd.Series(hex_type).value_counts()
area = cen.area / 1e6
area_by_type = {k: float(area[hex_type == k].sum()) for k in hex_by_type.index}
th = gpd.read_file(os.path.join(OLD, "ne_10m_admin_0_countries.shp"))
TH = gpd.GeoSeries([th[th.ISO_A3 == "THA"].geometry.iloc[0]], crs="EPSG:4326").to_crs(EQ)[0]
ne_area = TH.area / 1e6
priv_sites = int(n_by_type.get("ที่ดินส่วนบุคคล", 0))
priv_hex = int(hex_by_type.get("ที่ดินส่วนบุคคล", 0))
priv_area = area_by_type.get("ที่ดินส่วนบุคคล", 0.0)
log(f"ส่วนแบ่งจำนวนจุด: เอกชน {priv_sites}/{len(df)} = {priv_sites/len(df)*100:.1f}%")
log(f"ส่วนแบ่งจำนวน hex: เอกชน {priv_hex}/{len(hexes)} = {priv_hex/len(hexes)*100:.1f}%")
guard_area(sum(area_by_type.values()), ne_area, "day12-sum-vs-country", tol=0.02)
log(f"ผลรวมพื้นที่ hex {sum(area_by_type.values()):,.0f} ตร.กม. เทียบ NE {ne_area:,.0f} ตร.กม.")

# ภาคที่ "พึ่งที่ดินเอกชน" มากที่สุด
priv_share_region = {}
for reg in sorted(set(hex_region)):
    m = hex_region == reg
    priv_share_region[reg] = round(float((hex_type[m] == "ที่ดินส่วนบุคคล").mean() * 100), 1)
log("สัดส่วน hex ที่เอกชนให้บริการแยกภาค: " + json.dumps(priv_share_region, ensure_ascii=False))

# จังหวัดที่ทุก hex ถูกให้บริการโดยที่ดินเอกชนล้วน
prov_of_site = dict(zip(df["name"], df["province"].astype(str)))
hex_prov = np.array([prov_of_site[n] for n in near])
prov_priv = {}
for pv in sorted(set(hex_prov)):
    m = hex_prov == pv
    prov_priv[pv] = round(float((hex_type[m] == "ที่ดินส่วนบุคคล").mean() * 100), 1)
all_priv = sorted([k for k, v in prov_priv.items() if v == 100.0], key=lambda k: -int((hex_prov == k).sum()))
log(f"จังหวัดที่ hex ทั้งหมดถูกให้บริการโดยที่ดินเอกชน: {len(all_priv)} จังหวัด {all_priv[:5]}")

### 4. แผนที่

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 12.2), facecolor=BG); ax.set_facecolor(BG)
ax.set_aspect("equal"); ax.axis("off")
ax.add_collection(PolyCollection([np.asarray(p.exterior.coords) for p in
                                 (TH.geoms if TH.geom_type.startswith("Multi") else [TH])],
                                 facecolor="#111a2b", edgecolor="#4b5563", lw=0.8))
for k in hex_by_type.index:
    m = hex_type == k
    ax.add_collection(PolyCollection([np.asarray(poly.exterior.coords) for g in cen[m]
                                     for poly in (g.geoms if g.geom_type.startswith("Multi") else [g])],
                                     facecolor=COL.get(k, "#64748b"), edgecolor=COL.get(k, "#64748b"),
                                     lw=0.1, alpha=0.72))
site_xy = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude),
                           crs="EPSG:4326").to_crs(EQ)
for x, y in zip(site_xy.geometry.x, site_xy.geometry.y):
    ax.plot(x, y, marker="o", ms=3.4, color="#f8fafc", mec="#0d1522", mew=0.3, zorder=6)
bx, by, bx1, by1 = TH.bounds
ax.set_xlim(bx - 60_000, bx1 + 60_000); ax.set_ylim(by - 190_000, by1 + 40_000)
ax.set_title("Day 12 · Power — ใครถือครองฟ้ามืดของไทย" + chr(10) +
             f"hex 1,738 เซลล์ถูกจัดกลุ่มตามสภาพการถือครองที่ดินของจุดทะเบียนที่ใกล้ที่สุด",
             color=INK, fontsize=13, pad=26, fontweight="bold")
ax.text(0.5, 1.012, f"จุดทะเบียนที่เป็นที่ดินส่วนบุคคล {priv_sites} จาก {len(df)} แห่ง = {priv_sites/len(df)*100:.1f}% "
        f"ของจำนวนจุด · แต่ให้บริการพื้นที่ {priv_hex} จาก 1,738 hex = {priv_hex/len(hexes)*100:.1f}%",
        transform=ax.transAxes, ha="center", color="#fca5a5", fontsize=9.2)
row = 0
for k in hex_by_type.index:
    ax.plot([0.0], [-0.035 - 0.037*row], marker="s", color=COL.get(k, "#64748b"), ms=10,
            clip_on=False, transform=ax.transAxes, zorder=9)
    ax.text(0.030, -0.035 - 0.037*row,
            f"{k} — {int(hex_by_type[k])} hex · {area_by_type[k]:,.0f} ตร.กม. "
            f"({area_by_type[k]/ne_area*100:.1f}% ของประเทศ) · จุด {int(n_by_type.get(k, 0))} แห่ง",
            transform=ax.transAxes, fontsize=7.8, color="#cbd5e1", va="center")
    row += 1
fig.savefig(os.path.join(OUT, "day12_power_preview.png"), dpi=150, facecolor=BG, bbox_inches="tight")
log("เขียน day12_power_preview.png")

### 5. two-panel analysis

In [ ]:
fig2, (a1, a2) = plt.subplots(1, 2, figsize=(13.6, 5.4), facecolor=BG, gridspec_kw=dict(width_ratios=[1, 1]))
for a in (a1, a2): a.set_facecolor("#111a2b")
ks = list(hex_by_type.index)
a1.bar(range(len(ks)), [area_by_type[k]/1e3 for k in ks], color=[COL.get(k, "#64748b") for k in ks])
a1.set_xticks(range(len(ks)), ks, rotation=28, ha="right", fontsize=8)
a1.set_ylabel("พื้นที่ที่ให้บริการ (พัน ตร.กม.)", color="#cbd5e1")
a1.set_title("ฝั่งซ้าย: เนื้อที่ประเทศที่แต่ละสภาพการถือครอง \"ดูแล\"", color=INK, fontsize=11)
for i, k in enumerate(ks):
    a1.text(i, area_by_type[k]/1e3 + 3, f"{area_by_type[k]/ne_area*100:.1f}%", ha="center",
            fontsize=8, color="#cbd5e1")
order = sorted(priv_share_region.items(), key=lambda kv: -kv[1])
a2.barh(range(len(order)), [v for _, v in order], color=["#f4566a" if v >= 50 else "#38bdf8" for _, v in order])
a2.set_yticks(range(len(order)), [k for k, _ in order], fontsize=9)
a2.invert_yaxis()
a2.set_xlabel("สัดส่วน hex ในภาคที่จุดเอกชนให้บริการ (%)", color="#cbd5e1")
a2.set_title("ฝั่งขวา: ภูมิภาคไหนพึ่งที่ดินเอกชนมากที่สุด", color=INK, fontsize=11)
for i, (k, v) in enumerate(order):
    a2.text(v + 1.2, i, f"{v:.1f}%", va="center", fontsize=8, color="#cbd5e1")
for a in (a1, a2):
    for sp in a.spines.values(): sp.set_color("#4b5563")
    a.tick_params(colors="#94a3b8"); a.grid(alpha=0.12, color="#4b5563")
fig2.tight_layout(); fig2.savefig(os.path.join(OUT, "day12_shares.png"), dpi=150, facecolor=BG)
log("เขียน day12_shares.png")

### 6. stats + geojson

In [ ]:
stat = {
    "hex_total": int(len(hexes)), "site_total": int(len(df)),
    "serving_sites": int(len(set(near))),
    "metric_definitions": {
        "share_of_sites": "จำนวนจุดทะเบียนจำแนกตามฟิลด์ type (สัดส่วนต่อ 83 จุด)",
        "share_of_hex": "จำนวน hex ที่จุดประเภทนั้นเป็นจุดที่ใกล้ที่สุด (ต่อ 1,738 เซลล์)",
        "share_of_area": "ผลรวม area_km2 ของ hex กลุ่มนั้น หารด้วยเนื้อที่ไทยตาม Natural Earth",
        "region_private_share": "สัดส่วน hex ในภาคนั้นที่จุดเอกชนให้บริการ",
        "note_two_shares": "51.8% ของ *จำนวนจุด* กับ 39.6% ของ *พื้นที่* เป็นคนละตัวเลข ห้ามพูดปน"},
    "sites_by_holder": {k: int(v) for k, v in n_by_type.items()},
    "hex_by_holder": {k: int(v) for k, v in hex_by_type.items()},
    "area_by_holder_km2": {k: round(v, 1) for k, v in area_by_type.items()},
    "area_share_of_country_pct": {k: round(v/ne_area*100, 2) for k, v in area_by_type.items()},
    "private": {"sites": priv_sites, "sites_pct": round(priv_sites/len(df)*100, 1),
                "hex": priv_hex, "hex_pct": round(priv_hex/len(hexes)*100, 1),
                "area_km2": round(priv_area, 1), "area_pct": round(priv_area/ne_area*100, 2)},
    "reference_areas": {"sum_of_hex_area_km2": round(float(sum(area_by_type.values())), 1),
                        "natural_earth_thailand_km2": round(float(ne_area), 1)},
    "region_private_share_pct": priv_share_region,
    "provinces_served_only_by_private": {"count": len(all_priv), "list": all_priv},
    "validation": {"nearest_field_agreement_pct": round(agree*100, 2),
                   "hexes_disagreeing": int((recheck != near).sum()),
                   "cause": "polygon centroid ของเซลล์ที่ถูก clip ≠ ศูนย์กลาง H3 cell ที่ Day 3 ใช้"},
    "cross_reference_day10": {"sum_of_nominal_hex_area_km2": 515023.8,
        "why": "โพสต์นี้เทียบกับบั๊กที่จับได้ใน Day 10 (บวก attribute area แล้วได้มากกว่าประเทศ)"},
    "external_facts": {"thailand_provinces_total": 77,
                       "source": "ข้อมูลภายนอก (จำนวนจังหวัดทางการ) ไม่ใช่จากไฟล์ในเครื่อง"},
    "note": "การจัดกลุ่ม hex ใช้หลัก 'จุดทะเบียนที่ใกล้ที่สุด' เหมือน Day 3-4-10-11 "
            "จึงเป็นแผนที่ *ขอบเขตบริการ* ไม่ใช่ *สภาพการถือครองที่ดินจริงของผืนฟ้านั้น*"}
json.dump(stat, open(os.path.join(OUT, "day12_stats.json"), "w"), ensure_ascii=False, indent=1)
gj = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "properties": {"nearest_site": str(near[i]), "holder": str(hex_type[i]),
                                       "region_th": str(hex_region[i]), "km_nearest": float(km_dark[i])},
     "geometry": mapping(h)} for i, h in enumerate(hexes)]}
json.dump(gj, open(os.path.join(OUT, "day12_hex_holders.geojson"), "w"), ensure_ascii=False)
log(f"เขียน stats + geojson · เอกชน: {stat['private']}")
log(f"รวมเวลา {time.time()-T0:.1f} วิ")